# REF4 127 — Anchor-Invariant R Residual (NVIDIA L4)

Validation-only notebook. It must not read test data, fit production models, or build a submission ZIP.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%pip install -q catboost==1.2.10

In [ ]:
from pathlib import Path
import hashlib, json, os, platform, shutil, subprocess, sys, time, zipfile

EXPERIMENT_ID = 'REF4-ANCHOR-INVARIANT-R-RESIDUAL-L4-127'
DRIVE_ROOT = Path('/content/drive/MyDrive/LG aimer/L4_EXPERIMENTS') / EXPERIMENT_ID
INPUT_ROOT = DRIVE_ROOT / 'input'
CODE_ZIP = INPUT_ROOT / 'code/REF4_127_L4_CODE.zip'
TRAIN = INPUT_ROOT / 'data/train.csv'
ANCHOR = INPUT_ROOT / 'anchor/oof_predictions.csv'
MANIFEST = DRIVE_ROOT / 'manifest/SHA256SUMS.input'
WORK_ROOT = Path('/content/ref4_127_code')

gpu_name = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True
).strip()
if 'L4' not in gpu_name.upper():
    raise RuntimeError(f'BLOCKED_WRONG_ACCELERATOR: expected NVIDIA L4, got {gpu_name!r}')
print('GPU=', gpu_name)
print('Python=', sys.version)
print('Platform=', platform.platform())
print('CPU count=', os.cpu_count())
print('Drive root=', DRIVE_ROOT)

In [ ]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

required = [TRAIN, ANCHOR, CODE_ZIP, MANIFEST, INPUT_ROOT / 'code/audit_contract.json']
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError({'missing': missing})

records = []
for line in MANIFEST.read_text(encoding='utf-8').splitlines():
    if not line.strip():
        continue
    expected, relative = line.split(None, 1)
    path = DRIVE_ROOT / relative.strip()
    actual = sha256(path) if path.exists() else None
    records.append({'path': relative.strip(), 'expected': expected, 'actual': actual, 'ok': actual == expected})
if not records or not all(record['ok'] for record in records):
    raise RuntimeError({'status': 'BLOCKED_INPUT_HASH', 'records': records})
print(f'Input SHA verified: {sum(r["ok"] for r in records)}/{len(records)}')

In [ ]:
if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True)
with zipfile.ZipFile(CODE_ZIP) as archive:
    archive.testzip() is None or (_ for _ in ()).throw(RuntimeError('Corrupt code ZIP'))
    archive.extractall(WORK_ROOT)
RUNNER = WORK_ROOT / 'scripts/run_ref4_anchor_invariant_r_residual_l4_127.py'
if not RUNNER.exists():
    raise FileNotFoundError(RUNNER)
contract = json.loads((INPUT_ROOT / 'code/audit_contract.json').read_text(encoding='utf-8'))
runner_sha = sha256(RUNNER)
if runner_sha != contract['inputs']['runner_sha256']:
    raise RuntimeError({'status': 'BLOCKED_RUNNER_HASH', 'expected': contract['inputs']['runner_sha256'], 'actual': runner_sha})
print('Runner SHA=', runner_sha)

In [ ]:
def run_profile(profile: str, smoke: bool) -> None:
    result_dir = DRIVE_ROOT / 'results' / profile
    checkpoint_dir = DRIVE_ROOT / 'checkpoints' / profile
    log_dir = DRIVE_ROOT / 'logs'
    result_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    log_dir.mkdir(parents=True, exist_ok=True)
    result_json = result_dir / 'result.json'
    if result_json.exists():
        raise RuntimeError(f'BLOCKED_EXISTING_RESULT: {result_json}')
    command = [
        sys.executable, str(RUNNER),
        '--train', str(TRAIN),
        '--anchor-oof', str(ANCHOR),
        '--output-dir', str(result_dir),
        '--checkpoint-dir', str(checkpoint_dir),
        '--targets', '2022,2023,2024',
        '--device', 'gpu', '--devices', '0', '--no-cpu-fallback',
        '--iterations', '256', '--depth', '6', '--learning-rate', '0.025',
        '--bootstrap', '10000'
    ]
    if smoke:
        command += ['--smoke', '--no-resume']
    else:
        command += ['--resume']
    launch = {
        'experiment_id': EXPERIMENT_ID, 'profile': profile, 'gpu_name': gpu_name,
        'command': command, 'started_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
        'runner_sha256': sha256(RUNNER), 'status': 'RUNNING'
    }
    launch_path = log_dir / f'{profile}_last_launch.json'
    launch_path.write_text(json.dumps(launch, indent=2), encoding='utf-8')
    log_path = log_dir / f'{profile}.log'
    with log_path.open('w', encoding='utf-8') as log_stream:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log_stream.write(line)
            log_stream.flush()
        return_code = process.wait()
    launch['finished_utc'] = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())
    launch['return_code'] = return_code
    launch['status'] = 'COMPLETE' if return_code == 0 else 'FAILED'
    launch['log_sha256'] = sha256(log_path)
    launch_path.write_text(json.dumps(launch, indent=2), encoding='utf-8')
    if return_code != 0:
        raise RuntimeError(f'{profile} failed with return code {return_code}')
    if not result_json.exists():
        raise RuntimeError(f'{profile} ended without result.json')
    print(json.dumps(json.loads(result_json.read_text()), indent=2))

## Smoke
This checks execution only. Never use smoke metrics for model selection.

In [ ]:
run_profile('smoke_l4', smoke=True)

## Full strict-forward run
Run only after the smoke cell completed. The 2024 gate is evaluated once with the pre-locked contract.

In [ ]:
run_profile('full_l4', smoke=False)

In [ ]:
inventory = []
for path in sorted(DRIVE_ROOT.rglob('*')):
    if path.is_file() and path.name != 'output_sha256.json':
        inventory.append({'path': str(path.relative_to(DRIVE_ROOT)), 'size': path.stat().st_size, 'sha256': sha256(path)})
audit_dir = DRIVE_ROOT / 'audit'
audit_dir.mkdir(parents=True, exist_ok=True)
inventory_path = audit_dir / 'output_sha256.json'
inventory_path.write_text(json.dumps({'files': inventory}, indent=2), encoding='utf-8')
print('Inventory files=', len(inventory), 'SHA=', sha256(inventory_path))
print('Maximum status before independent local audit: PERFORMANCE_GATE_PASS_UNAUDITED')